# 客户表清洗第一步：两个客户 ID、主键与订单关联

本次先理解数据关系，读取和检查原始值。城市、州和邮编的规范处理留到下一步。

一行是一条 customer_id 对应的客户记录。customer_unique_id 用于跨订单识别同一客户标识，因此同一个值可以出现在多行。

| 字段 | 含义与用途 |
| --- | --- |
| customer_id | 与订单表连接的客户记录 ID |
| customer_unique_id | 跨订单客户标识，后续按它做客户分组 |
| customer_zip_code_prefix | 邮编前五位，保留为字符串以保留前导零 |
| customer_city | 客户所在城市 |
| customer_state | 客户所在州或联邦区的缩写 |

字段解释参照原始目录的中文数据字典。记录数、不同客户标识数与符合复购口径的客户数是不同概念。
customer_unique_id就相当于客户的唯一Id，跟订单没关系，只要是这个顾客买任何东西都是这个id
customer_id就算是一个客户针对不同的订单也有不同的id

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'

def read_raw(filename):
    # 所有字段先读成字符串：ID 与邮编都是标识，不应因外观像数字而自动转型。
    # keep_default_na=False 保留字面文本；这里只有空字段识别为缺失。
    return pd.read_csv(
        data_dir / filename,
        dtype=str,
        encoding='utf-8-sig',
        keep_default_na=False,
        na_values=['']
    )

# 保留两张原始表；后面的关联会生成新表。
customers_raw = read_raw('olist_customers_dataset.csv')
orders_raw = read_raw('olist_orders_dataset.csv')

print('客户表行数、列数：', customers_raw.shape)
customers_raw.head(3)

客户表行数、列数： (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP


In [2]:
# 先看字段类型及非空数量；当前暂不清洗地区和邮编。
customers_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  object
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: object(5)
memory usage: 3.8+ MB


## 1. 哪一列能作为当前表主键

先检查两个 ID 的非空和唯一性。customer_unique_id 名字里有 unique，不代表它在这张表里每行都不同。

主键需要非空且唯一。单个客户标识对应多条订单关联记录时，不能直接按该标识删行。

In [3]:
customer_key_check = pd.Series({
    '客户记录行数': len(customers_raw),
    'customer_id不同值数': customers_raw['customer_id'].nunique(),
    'customer_unique_id不同值数': customers_raw['customer_unique_id'].nunique(),
    'customer_id缺失行数': customers_raw['customer_id'].isna().sum(),
    'customer_unique_id缺失行数': customers_raw['customer_unique_id'].isna().sum(),
    'customer_id空白字符串行数': (
        customers_raw['customer_id'].fillna('').str.strip().eq('')
        & customers_raw['customer_id'].notna()
    ).sum(),
    'customer_unique_id空白字符串行数': (
        customers_raw['customer_unique_id'].fillna('').str.strip().eq('')
        & customers_raw['customer_unique_id'].notna()
    ).sum(),
    'customer_id重复行数_首条之后': customers_raw.duplicated(['customer_id']).sum(),
    'customer_unique_id重复行数_首条之后': (
        customers_raw.duplicated(['customer_unique_id']).sum()
    ),
    '整行重复数_首条之后': customers_raw.duplicated().sum()
})

customer_key_check

客户记录行数                         99441
customer_id不同值数                99441
customer_unique_id不同值数         96096
customer_id缺失行数                    0
customer_unique_id缺失行数             0
customer_id空白字符串行数                 0
customer_unique_id空白字符串行数          0
customer_id重复行数_首条之后               0
customer_unique_id重复行数_首条之后     3345
整行重复数_首条之后                         0
dtype: int64

## 2. 查看两个 ID 的映射关系

groupby(customer_unique_id) 是把同一跨订单客户标识对应的记录放在一组；nunique 统计它对应几个不同 customer_id。

这里统计的是 ID 的映射，不是最终复购结果。有效购买状态、时间范围与观察窗口还未筛选。

In [4]:
# 正向：每个客户记录 ID 对应几个跨订单客户标识？
unique_ids_per_customer_id = (
    customers_raw.groupby('customer_id')['customer_unique_id'].nunique()
)

# 反向：每个跨订单客户标识对应几个不同的客户记录 ID？
customer_ids_per_unique = (
    customers_raw.groupby('customer_unique_id')['customer_id'].nunique()
)

pd.Series({
    '对应多个unique_id的customer_id数': (
        unique_ids_per_customer_id.gt(1).sum()
    ),
    '对应多个customer_id的unique_id数': (
        customer_ids_per_unique.gt(1).sum()
    ),
    '单个unique_id最多对应customer_id数': customer_ids_per_unique.max()
})

对应多个unique_id的customer_id数        0
对应多个customer_id的unique_id数     2997
单个unique_id最多对应customer_id数      17
dtype: int64

In [5]:
# 选一个恰好对应两条客户记录的客户标识，观察两个 customer_id。
example_unique_id = customer_ids_per_unique.loc[
    customer_ids_per_unique.eq(2)
].index[0]

customers_raw.loc[
    customers_raw['customer_unique_id'].eq(example_unique_id),
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
     'customer_city', 'customer_state']
]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
35608,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA


## 3. 按 customer_id 关联订单

订单原表提供 customer_id。先按它关联客户表，才能把 customer_unique_id、城市和州带入订单。

当前两张原始表的 customer_id 均唯一，所以本次用 one_to_one 检查观察到的关系。相同 customer_unique_id 可以出现在不同客户记录中，不影响按 customer_id 连接。

In [6]:
# 连接前检查客户表主键，防止右侧重复键把订单行数放大。
if (
    customers_raw['customer_id'].isna().any()
    or customers_raw['customer_id'].str.strip().eq('').any()
    or customers_raw['customer_id'].duplicated().any()
):
    raise ValueError('客户表主键缺失、空白或重复，请先核查')

# 左连接不会保留仅在客户表出现的记录，所以两侧覆盖分别检查。
orders_without_customer = (
    ~orders_raw['customer_id'].isin(customers_raw['customer_id'])
).sum()

customer_records_without_order = (
    ~customers_raw['customer_id'].isin(orders_raw['customer_id'])
).sum()

print('无法匹配客户表的订单数：', orders_without_customer)
print('没有对应订单的客户记录数：', customer_records_without_order)

if orders_without_customer > 0:
    raise ValueError('存在缺少客户记录的订单，请先核查')

# 使用订单主表做左表，保留全部订单。
# validate 只检查连接键的关系；它不负责去重。
orders_with_customers = orders_raw.merge(
    customers_raw,
    on='customer_id',
    how='left',
    validate='one_to_one',
    indicator='customers_match'
)

print('关联后订单行数：', len(orders_with_customers))
orders_with_customers['customers_match'].value_counts()

无法匹配客户表的订单数： 0
没有对应订单的客户记录数： 0
关联后订单行数： 99441


customers_match
both          99441
left_only         0
right_only        0
Name: count, dtype: int64

In [7]:
# customer_unique_id 已被带入订单，后续可以按它汇总有效购买。
# 这里先查看关联结果，暂不计算复购率。
orders_with_customers[
    ['order_id', 'order_status', 'customer_id', 'customer_unique_id',
     'customer_zip_code_prefix', 'customer_city', 'customer_state']
].head()

,order_id,order_status,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP


## 对照基线与下一步

预期客户表 99,441 行；customer_id 不同值 99,441，customer_unique_id 不同值 96,096。两种 ID 均非空；前者没有重复，后者首条之后重复 3,345 行。

2,997 个 customer_unique_id 对应多个 customer_id，最多对应 17 个。当前只能称为“对应多条客户记录的客户标识”，不能直接当作最终复购客户数。

订单与客户关联后保持 99,441 行，全部匹配。邮编保留字符串，如 01151 的前导零需要保留。下一步检查邮编格式以及城市、州字段。

## 我的理解

1. 客户表有多少条记录？customer_id、customer_unique_id 的不同值数分别是多少？

我的回答：客户表有99441行，customer_id不同值有99441个，customer_unique_id不同值有96096条。

2. 哪一列能作为当前客户表的主键？customer_unique_id 为什么不能单独作为本表主键？

我的回答：customer_id当作主键，customer_unique_id有重复值，所以不能单独作为主键。

3. 同一个 customer_unique_id 对应多条记录，能否直接按它去重？这样做可能影响什么？

我的回答：不能直接去重，同一个 customer_unique_id可能对应不同customer_id，对应不同的order_id。

4. 与订单表连接用哪个 ID？后续识别同一客户的多次购买用哪个 ID？为什么当前的 2,997 还不能直接称为最终复购客户数？

我的回答：与订单表连接用customer_id；识别同一客户多次购买用customer_unique_id；复购客户数要先明确有效订单状态、分析时间范围和复购规则。


In [8]:
# 原表继续保留；后面的处理写入工作副本。
customers_clean = customers_raw.copy()

location_cols = [
    'customer_zip_code_prefix',
    'customer_city',
    'customer_state'
]

location_check = {}

for col in location_cols:
    original = customers_raw[col]

    # 去掉首尾空白；不删除城市名称中间正常的空格。
    normalized = original.str.strip()

    if col == 'customer_city':
        normalized = normalized.str.lower()
    elif col == 'customer_state':
        normalized = normalized.str.upper()

    customers_clean[col] = normalized

    location_check[col] = {
        '原始缺失数': original.isna().sum(),
        '非空但只有空白字符数': (
            original.notna() & normalized.eq('')
        ).sum(),
        '规范后发生变化的记录数': (
            original.notna() & original.ne(normalized)
        ).sum()
    }

location_check = pd.DataFrame(location_check).T
location_check

,原始缺失数,非空但只有空白字符数,规范后发生变化的记录数
customer_zip_code_prefix,0,0,0
customer_city,0,0,0
customer_state,0,0,0


In [9]:
# 邮编是标识，保留字符串和前导零，例如 01151。
zip_text = customers_clean['customer_zip_code_prefix']
state_text = customers_clean['customer_state']

# fullmatch 要求整个字符串符合规则。
# na=False 表示缺失值也不能通过格式检查。
customers_clean['flag_zip_format_invalid'] = (
    ~zip_text.str.fullmatch('[0-9]{5}', na=False)
)

customers_clean['flag_state_format_invalid'] = (
    ~state_text.str.fullmatch('[A-Z]{2}', na=False)
)

format_check = pd.Series({
    '邮编格式不符记录数': (
        customers_clean['flag_zip_format_invalid'].sum()
    ),
    '州标签格式不符记录数': (
        customers_clean['flag_state_format_invalid'].sum()
    ),
    '以0开头的邮编记录数': zip_text.str.startswith('0').sum(),
    '不同州标签数': state_text.nunique()
})

format_check

邮编格式不符记录数         0
州标签格式不符记录数        0
以0开头的邮编记录数    23995
不同州标签数           27
dtype: int64

In [10]:
# 对每个城市名称，统计它出现在几个不同的州。
states_per_city = (
    customers_clean.groupby('customer_city')['customer_state']
    .nunique()
)

multi_state_cities = states_per_city.loc[
    states_per_city.gt(1)
]

print('出现在多个州的城市名称数：', len(multi_state_cities))

example_city = multi_state_cities.index[0]

# 这里只对展示用的“城市、州”组合去重，不修改客户表。
customers_clean.loc[
    customers_clean['customer_city'].eq(example_city),
    ['customer_city', 'customer_state']
].drop_duplicates().sort_values('customer_state')

出现在多个州的城市名称数： 163


,customer_city,customer_state
5461,agua branca,AL
65688,agua branca,PB
12371,agua branca,PI
